In [1]:
import os
from langchain_huggingface import HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA

HF_TOKEN=os.environ.get("HF_TOKEN") or ""

# --- 1. Rebuild the LLM with Stability Fixes ---
hf = HuggingFaceEndpoint(
    # Swap to the Instruct version! It understands Q&A formatting and knows when to stop.
    repo_id="mistralai/Mistral-7B-Instruct-v0.3", 
    huggingfacehub_api_token=HF_TOKEN,
    task="text-generation",
    temperature=0.1,
    max_new_tokens=500,
    # Add a generous timeout so the server doesn't sever the connection early
    timeout=120 
)

# --- 2. Rebuild the Prompt Template ---
prompt_template="""
Use the following piece of context to answer the question asked.
Please try to provide the answer only based on the context.

Context:
{context}

Question: {question}

Helpful Answer:
"""
prompt = PromptTemplate(template=prompt_template, input_variables=["context","question"])

# --- 3. Rebuild and Run the Chain ---
retrievalQA = RetrievalQA.from_chain_type(
    llm=hf,
    chain_type="stuff",
    # Assuming your FAISS retriever is still loaded in memory from the cell above
    retriever=retriever, 
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt}
)

query = """DIFFERENCES IN THE UNINSURED RATE BY STATE IN 2022"""

print("--- Sending request to Hugging Face... This may take a moment. ---\n")

# Run the query
result = retrievalQA.invoke({"query": query})

print(result['result'])

ImportError: cannot import name 'ModelProfile' from 'langchain_core.language_models' (/Users/adi/Coding/GenAI/GenAi_Krish_Naik/ByMe/venv/lib/python3.13/site-packages/langchain_core/language_models/__init__.py)